In [1]:
import sys
sys.path.append('/home/jovyan/work/Similarity-Aware-Label-Smoothing')


In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import ResNet18, CifarDenseNet121, TinyEfficientNet
# from metrics import calibration_errors, nll_loss, accuracy
import random

## Hyperparams

In [29]:
seed = 14
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [30]:
dataset = "cifar100"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler(device="cuda")
lr = 0.1
epochs = 200
num_classes = 100
epsilon = 0.1
K = 10

/var/folders/r3/6v_h8z6936vbpxsh4znh6mbm0000gn/T/ipykernel_21678/486777333.py:3: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler(device="cuda")


## Training Utils

### Label Smoothing

In [31]:
!ls

README.md                       main.ipynb
__pycache__                     metrics.py
confidence.py                   models
data                            models.py
dataset_utils.py                output.png
eval.ipynb                      results.pth
figs_calibration_vs_epsilon     results.xlsx
figs_ece_vs_epsilon_equalspaced scores
figs_k_ablations                vae.ipynb


In [32]:
path = f"scores/cifar100/4_128_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=K, tau=1.2):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(-class_dists / tau, dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


In [34]:
labels = torch.arange(20, device=device)

vae_all = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
ds = CIFAR100(root="./data", train=True, download=True)
name_to_idx = {name: i for i, name in enumerate(ds.classes)}
idx_to_name = {i: name for name, i in name_to_idx.items()}

y = name_to_idx["rose"]
vae = vae_all[y].detach().cpu()

ce = torch.zeros(num_classes)
ce[y] = 1.0

# --- neighbors from SSA/VAE-LS ---
nz = (vae > 0).nonzero(as_tuple=False).view(-1).tolist()
nz = [i for i in nz if i != y]
nz = sorted(nz, key=lambda i: float(vae[i]), reverse=True)[:10]

/Users/nihar/Desktop/CS 444/Prep/444venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [35]:
labels_txt = [idx_to_name[i].replace("_", " ") for i in nz]
labels_txt


['tulip',
 'poppy',
 'orchid',
 'lobster',
 'crab',
 'boy',
 'baby',
 'man',
 'girl',
 'woman']

## Training Loop

In [105]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):
    classwise_target = classwise_target.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            targets = classwise_target[y]

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(x)
                loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        if epoch % 10 == 9 or epoch >= 195:
            test_acc = accuracy(model, test_loader)
            print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [106]:
classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target[0])

tensor([0.9000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0189,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0188, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0171,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0260, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0194, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000], device='cuda:0')


In [107]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

In [108]:
model = CifarDenseNet121(num_classes=num_classes).to(device)
model = torch.compile(model, mode="max-autotune")
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
)

  0%|          | 0/391 [00:00<?, ?it/s]

Epoch 10/200 | Loss: 2.2608 | Test Acc: 0.4364 | Top-5 Test Acc: 0.7441


Epoch 20/200 | Loss: 2.0277 | Test Acc: 0.4717 | Top-5 Test Acc: 0.7799


Epoch 30/200 | Loss: 1.9225 | Test Acc: 0.4967 | Top-5 Test Acc: 0.7938


Epoch 40/200 | Loss: 1.8482 | Test Acc: 0.5242 | Top-5 Test Acc: 0.8119


...

Epoch 196/200 | Loss: 0.5167 | Test Acc: 0.6676 | Top-5 Test Acc: 0.8444


Epoch 197/200 | Loss: 0.5165 | Test Acc: 0.6673 | Top-5 Test Acc: 0.8444


Epoch 198/200 | Loss: 0.5167 | Test Acc: 0.6650 | Top-5 Test Acc: 0.8434


Epoch 199/200 | Loss: 0.5168 | Test Acc: 0.6662 | Top-5 Test Acc: 0.8436


Epoch 200/200 | Loss: 0.5166 | Test Acc: 0.6678 | Top-5 Test Acc: 0.8425


In [109]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x)

        logits_list.append(logits.float())  
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)

print("ECE:", calibration_errors(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))

test_acc = accuracy(model, test_loader)  
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


ECE: (0.07979477941989899, 0.20406073331832886)
NLL: 1.4542357921600342
Top-1 Test Acc: 0.6678 | Top-5 Test Acc: 0.8425


In [110]:
PATH = f"vae4_{'0'+str(epsilon).removeprefix("0.")}_{K}_{seed}.pth"
# PATH = f"ls_{'0'+str(epsilon).removeprefix("0.")}_{seed}.pth"
torch.save(model._orig_mod.state_dict(), PATH)